# 7E Run a Model Bake-Off

**Purpose:** Compare gateway aliases on the same small, versioned task set, while keeping the evidence and limitations visible.

**What we're doing**

Plan the request budget, run deterministic checks over a handful of responses, and export every attempt, including failures, for inspection. Optionally visualize task pass rate against client-observed latency.

**Why we're doing it**

Choosing a model from one impressive answer hides reliability, formatting failures, routing, and request overhead. A repeatable local harness makes a more useful starting point, without pretending that four questions establish a general ranking.

**What to look for**

* Identical system prompts, questions, output caps, and temperature settings across aliases.
* Requested aliases kept separate from the backend model reported by the gateway.
* Missing usage stays unknown; failed requests stay in the results.
* Client-observed request latency is not model throughput, and token counts are not prices.

**Output:** A run-specific directory containing `results.jsonl`, spreadsheet-safe `results.csv`, and optionally `quality_latency.png`.

> **Before you start:** Finish the [VS Code introduction](07_Running_VS_Code.ipynb) and [Section 2 gateway lab](../02_LLMGateway/02a_Use_the_LLM_Gateway.ipynb). Use Python 3.10+ and two distinct aliases permitted by your profile. They may map to the same backend, so verify the profile configuration before interpreting a comparison. A local GPU is not required by the harness.

This is a guided terminal lab. Run commands in the **remote IDE terminal from the repository root**. Offline tests and dry runs need no gateway credentials.


## 7E.1 Prepare a Reproducible Workspace

Reuse the environment from the earlier VS Code labs, or create it if needed:

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install -r 07_VSCode/requirements.txt
python -m pip install 'scikit-learn>=1.5,<2'
python --version
git rev-parse HEAD
```

Scikit-learn is for the combined RAG/evaluation offline test suite, not the evaluation runner itself. Optional charts require:

```bash
python -m pip install 'matplotlib>=3.9,<4'
```

Open [`evaluate_models.py`](evaluate_models.py), [`eval_questions.json`](eval_questions.json), and [`gateway_client.py`](gateway_client.py). Record the repository commit and Python version with your run notes. Avoid publishing a package list if your environment contains private package locations.


## 7E.2 Read the Questions and Scoring Rules

The bundled suite, `klique-mini-eval-v1`, contains four deliberately narrow tasks:

| Question | Check | What it exercises |
|---|---|---|
| Arithmetic | Exact `102` | Simple correctness and concise instruction following |
| JSON format | Parse JSON and compare the expected object | Machine-readable output without Markdown fences |
| Grounded answer | Exact supplied workspace path | Use a fact in the prompt |
| Abstention | Exact `UNKNOWN` | Do not invent a token absent from the prompt |

No real private token or workspace location is included. Exact matching strips outer whitespace but remains case-sensitive. JSON scoring tolerates property order but rejects extra properties, fences, and wrong types. A response can be useful to a person and still fail a strict formatting test; read failed answers before concluding the model is unsuitable.

The runner records a human-readable suite version and the SHA-256 of the entire question file. It also records `klique-eval-system-v1` and a hash of the system prompt. If you change a question, expected answer, or scoring intent, increment the suite version. If you change the system prompt, increment `PROMPT_VERSION` too. Retain the exact source files and commit for reproducibility.

> **Checkpoint:** Before calling models, explain what each pass means and what it does not measure. There is no LLM judge and no claim of broad reasoning, safety, or production quality.


## 7E.3 Plan the Budget Without Calling a Model

```bash
python 07_VSCode/evaluate_models.py --models alias-a alias-b
```

Expected: `Plan: 8 requests; at most 2048 requested output tokens; no automatic retries.` Then a dry-run message. Placeholder aliases are intentional here. No configuration is loaded and no output directory is created in dry-run mode.

The harness is sequential. One repeat over two aliases and four questions makes eight requests. Every request uses `temperature=0` and `max_tokens=256`. At most three repeats and 24 total calls are allowed; excess plans are rejected before any request. Two aliases with `--repeats 3` use the full 24-call cap. Alias order reverses on alternating repeats to reduce, not eliminate, ordering effects.

The output-token cap is **not a total-token or monetary cap**. Prompt tokens, provider-specific reasoning accounting, minimum charges, and gateway policy may affect actual usage and cost. Confirm the profile budget and provider pricing before live execution. Temperature zero is not a guarantee of deterministic responses across hardware, routing, or serving updates.


## 7E.4 Run One Small Live Comparison

Configure the ignored repository-root `.env` using the shared setup in the [AI playground](07b_Build_a_Governed_AI_Playground.ipynb). Set the gateway URL ending in `/llm`, token, profile, and `KLIQUE_MODEL_ALIASES`. Use the aliases from your profile, not underlying endpoint names. Do not print `.env` or tokens.

Replace the aliases below with two configured names and choose a **new** output directory:

```bash
python 07_VSCode/evaluate_models.py --models your-alias-a your-alias-b --live --output lab_outputs/bakeoff-first
```

`--live` explicitly authorizes this local command to send the bundled nonsensitive prompts through the gateway. The shared client supplies authentication and profile routing. No credential is inserted into a prompt or result. Failed requests are not automatically retried; each becomes an error row. Authentication, budget, rate-limit, ambiguous-limit, or transport errors halt further calls, with remaining planned rows marked skipped. Other errors allow the bounded run to continue.

Expected: eight planned rows saved, plus error and skipped counts. Existing output directories are refused before calls, so rerunning this exact command does not silently overwrite the original. For another intentional run, use a fresh directory name. An interrupted process may not reach export; do not assume a retry is free or that earlier calls did not occur. Check the gateway usage dashboard first.

> **Checkpoint:** Open both export files in VS Code. Eight rows means eight planned comparisons, not necessarily eight attempts or successes. Check status for skipped rows.

> **Screenshot placeholder:** CSV preview with requested alias, status, pass result, latency and token fields. Keep credentials and private metadata outside the capture.


## 7E.5 Inspect All Attempts, Including Errors

Each row records UTC time, profile, requested alias, returned backend model if supplied, question ID, repeat, suite and prompt versions/hashes, generation settings, answer, pass result, status, finish reason, error kind/status, elapsed seconds, and any returned token counts.

JSONL preserves `null` for missing usage, missing model metadata, and unscored failures. CSV shows those values as blank cells. **Unknown is not zero.** A finish reason such as `length` can explain a truncated response; missing finish reasons remain unknown. Skipped rows have no call, score, usage, or latency. Error rows deliberately omit raw exception messages and response bodies, which could contain secrets. A status of `ok` means a text response was obtained, not that the check passed.

Client-observed `latency_seconds` spans the synchronous request, including network, queueing, routing, and generation. It is neither time-to-first-token nor tokens-per-second. With only a few observations, differences may reflect cold starts, load, caching, or route changes. Do not label a model “faster” from one sample without those qualifications.

For a summary, report all of these together:

* Actual attempts, error count, and skipped count per alias.
* Passes divided by **all actual attempts**, treating errors as non-passes and excluding skipped rows.
* Which individual questions failed and the raw answers that explain why.
* Latencies for successful requests separately from error latencies.
* Missing usage counts, requested alias, and reported backend model.

There are no fabricated dollar estimates or throughput metrics. Use verified pricing and gateway billing records for cost, not assumptions based on model names. The CSV prefixes formula-like strings with an apostrophe for spreadsheet safety; JSONL retains the original text and should not be executed or blindly pasted into formulas.


## 7E.6 Optional Plot and Controlled Experiment

After installing matplotlib, use a **new** run directory for a plotted run:

```bash
python 07_VSCode/evaluate_models.py --models your-alias-a your-alias-b --live --output lab_outputs/bakeoff-chart --plot
```

This command makes another eight live requests. `--plot` is not a read-only replot command. It writes the result files first, then a quality-versus-latency scatter plot. Each alias has a task pass rate (errors remain in the denominator, skipped rows do not) and mean client-observed latency over successful requests only. Aliases without successful responses are listed but not plotted; compare their failure counts separately. If matplotlib is missing, the results are still exported before plotting fails.

For a read-only plot of an existing run, use this terminal heredoc instead:

```bash
python - <<'PYCODE'
import json
import sys
from pathlib import Path
sys.path.insert(0, '07_VSCode')
from evaluate_models import plot_rows
folder = Path('lab_outputs/bakeoff-first')
rows = [json.loads(line) for line in (folder / 'results.jsonl').read_text().splitlines()]
plot_rows(rows, folder / 'quality_latency.png')
PYCODE
```

**Exercise:** Add two realistic, nonsensitive tasks from your application to a copy of the suite, with explicit expected outputs. Increment its version and use `--questions path/to/your-suite.json`. Hold prompts and model settings fixed while changing just the alias. Recompute the request count before using repeats. Manual qualitative review is appropriate for tasks that cannot honestly be reduced to these exact-match checks; do not disguise a subjective judgment as an objective score.


## 7E.7 Verify the Harness Offline

```bash
python -m unittest discover -s 07_VSCode -p test_rag_evaluation.py -v
```

Expected: all tests pass without credentials or external requests. Evaluation tests cover call limits before invocation, one row per planned comparison, including failures and skipped calls after a halt, omission of raw exception text, unknown usage, token recording, exact/JSON scoring, CSV formula neutralization, JSONL fidelity, and overwrite protection. RAG tests from 7C also run.

A mocked test proves the harness handles those cases, not that your gateway credentials or chosen model work. Live access and model behavior are separate checks. Never replace an actual error row with a mock success in a reported run.


## 7E.8 Save the Evidence and Stop Compute

Keep the suite, prompt version, code commit, exported rows, and environment notes together. Review result files before sharing because answers and profile/model identifiers may reveal internal details. Keep generated `lab_outputs/` files out of source commits unless you intentionally reviewed them for publication.

No web server or background process runs in this lab. Save intentional changes, stop the VS Code session, and verify stopped status. Serving deployments have independent lifecycles and should be stopped only when you own them and no longer need them.

## Troubleshooting

| Symptom | Check |
|---|---|
| Alias rejected locally | Include it in `KLIQUE_MODEL_ALIASES`; confirm the profile permits it. |
| All rows are 401 / 404 | Check token, gateway URL, profile and alias names. Local configuration does not grant gateway access. |
| Some rows are timeout / 502 / 503 | Inspect the backing deployment and usage before deliberately making another run. |
| Successful response, failed check | Read the answer against the exact scoring rule; extra prose or Markdown may be the failure. |
| Token fields blank | The provider did not return usable usage. Leave them unknown. |
| Backend model equals another alias's backend | Routing can map multiple aliases to the same deployment. Verify profile configuration. |
| Directory exists | Choose a new output path; never overwrite the evidence to hide a failed run. |
| Plot fails after export | Install optional matplotlib, then replot existing JSONL without new calls. |

## Wrap-Up

You have a bounded, inspectable comparison rather than a winner picked from a single anecdote. Use these results to select the next experiment, not as a universal model leaderboard.

**Back:** [Debug an agent](07d_Debug_an_Agent.ipynb)  
**Next:** [Train, stop, and resume](07f_Train_Stop_Restore_Continue.ipynb)  
**Index:** [README](../README.md)
